# NumPy 배열 계산: 범용 함수

지금까지 우리는 NumPy의 기본 너트와 볼트에 대해 논의했습니다. 다음 몇 장에서는 NumPy가 파이썬(Python) 데이터 과학(Data Science) 세계에서 그토록 중요한 이유, 즉 NumPy가 데이터 배열로 계산을 최적화하기 위한 쉽고 유연한 인터페이스를 제공하기 때문입니다.

NumPy 배열의 계산은 매우 빠를 수도 있고 매우 느릴 수도 있습니다.
이를 빠르게 만드는 핵심은 일반적으로 NumPy의 *범용 함수*(ufuncs)를 통해 구현되는 벡터화된 작업을 사용하는 것입니다.
이번 장에서는 배열 요소에 대한 반복 계산을 훨씬 더 효율적으로 만드는 데 사용할 수 있는 NumPy의 ufunc에 대한 필요성을 설명합니다.
그런 다음 NumPy 패키지에서 사용할 수 있는 가장 일반적이고 유용한 산술 ufunc를 소개합니다.

## 루프의 느림

파이썬(Python)의 기본 구현(C파이썬(Python)이라고도 함)은 일부 작업을 매우 느리게 수행합니다.
이는 부분적으로 언어의 역동적이고 해석된 특성 때문입니다. 유형은 유연하므로 C 및 Fortran과 같은 언어에서와 같이 작업 순서를 효율적인 기계어 코드로 컴파일할 수 없습니다.
최근 이 약점을 해결하려는 다양한 시도가 있었습니다. 잘 알려진 예로는 파이썬(Python)의 JIT(Just-In-Time) 컴파일 구현인 [PyPy 프로젝트](http://pypy.org/)가 있습니다. 파이썬(Python) 코드를 컴파일 가능한 C 코드로 변환하는 [Cython 프로젝트](http://cython.org); 파이썬(Python) 코드 조각을 빠른 LLVM 바이트코드로 변환하는 [Numba 프로젝트](http://numba.pydata.org/).
이들 각각에는 장단점이 있지만 세 가지 접근 방식 중 어느 것도 아직 표준 C파이썬(Python) 엔진의 도달 범위와 인기를 능가하지 못했다고 해도 과언이 아닙니다.

파이썬(Python)의 상대적인 느린 속도는 일반적으로 많은 작은 작업이 반복되는 상황에서 나타납니다. 예를 들어 배열을 반복하여 각 요소에 대해 작동합니다.
예를 들어 값의 배열이 있고 각각의 역수를 계산하고 싶다고 가정해 보겠습니다.
간단한 접근 방식은 다음과 같습니다.

In [1]:
import numpy as np
rng = np.random.default_rng(seed=1701)

def compute_reciprocals(values):
    output = np.empty(len(values))
    for i in range(len(values)):
        output[i] = 1.0 / values[i]
    return output
        
values = rng.integers(1, 10, size=5)
compute_reciprocals(values)

array([0.11111111, 0.25      , 1.        , 0.33333333, 0.125     ])

이 구현은 아마도 C 또는 Java 배경을 가진 사람에게는 매우 자연스럽게 느껴질 것입니다.
그러나 대규모 입력에 대한 이 코드의 실행 시간을 측정하면 이 작업이 매우 느리다는 것을 확인할 수 있습니다. 아마도 놀랍게도 그렇습니다!
우리는 이것을 I파이썬(Python)의 `%timeit` 매직으로 벤치마킹할 것입니다([프로파일링 및 타이밍 코드](01.07-Timing-and-Profiling.ipynb)에서 논의됨):

In [2]:
big_array = rng.integers(1, 100, size=1000000)
%timeit compute_reciprocals(big_array)

2.61 s ± 192 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


수백만 개의 연산을 계산하고 결과를 저장하는 데 몇 초가 걸립니다!
휴대폰의 처리 속도가 기가플롭스(즉, 초당 수십억 번의 수치 연산)로 측정되는 경우에도 이는 거의 터무니없이 느린 것처럼 보입니다.
여기서 병목 현상은 작업 자체가 아니라 C파이썬(Python)이 루프의 각 주기에서 수행해야 하는 유형 검사 및 함수 디스패치라는 것이 밝혀졌습니다.
역수가 계산될 때마다 파이썬(Python)은 먼저 객체의 유형을 검사하고 해당 유형에 사용할 올바른 함수의 동적 조회를 수행합니다.
대신 컴파일된 코드로 작업한다면 이 유형 사양은 코드가 실행되기 전에 알려지고 결과는 훨씬 더 효율적으로 계산될 수 있습니다.

## Ufuncs 소개

많은 유형의 작업에 대해 NumPy는 이러한 종류의 정적으로 유형이 지정되고 컴파일된 루틴에 대한 편리한 인터페이스를 제공합니다. 이를 *벡터화된* 작업이라고 합니다.
여기서 요소별 나누기와 같은 간단한 연산의 경우 벡터화는 배열 객체에 직접 파이썬(Python) 산술 연산자를 사용하는 것만큼 간단합니다.
이 벡터화된 접근 방식은 NumPy의 기반이 되는 컴파일된 레이어에 루프를 푸시하여 훨씬 더 빠른 실행을 제공하도록 설계되었습니다.

다음 두 작업의 결과를 비교하십시오.

In [3]:
print(compute_reciprocals(values))
print(1.0 / values)

[0.11111111 0.25       1.         0.33333333 0.125     ]
[0.11111111 0.25       1.         0.33333333 0.125     ]


큰 배열의 실행 시간을 살펴보면 파이썬(Python) 루프보다 훨씬 빠르게 완료된다는 것을 확인할 수 있습니다.

In [4]:
%timeit (1.0 / big_array)

2.54 ms ± 383 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


NumPy의 벡터화된 작업은 ufuncs를 통해 구현됩니다. ufuncs의 주요 목적은 NumPy 배열의 값에 대해 반복 작업을 빠르게 실행하는 것입니다.
Ufuncs는 스칼라와 배열 사이의 연산을 보기 전에는 매우 유연하지만 두 배열 사이에서도 작동할 수 있습니다.

In [5]:
np.arange(5) / np.arange(1, 6)

array([0.        , 0.5       , 0.66666667, 0.75      , 0.8       ])

그리고 ufunc 연산은 1차원 배열에만 국한되지 않습니다. 다차원 배열에서도 작동할 수 있습니다.

In [6]:
x = np.arange(9).reshape((3, 3))
2 ** x

array([[  1,   2,   4],
       [  8,  16,  32],
       [ 64, 128, 256]])

ufunc를 통한 벡터화를 사용한 계산은 파이썬(Python) 루프를 사용하여 구현된 계산보다 거의 항상 더 효율적입니다. 특히 배열의 크기가 커질수록 더욱 그렇습니다.
NumPy 스크립트에서 이러한 루프를 볼 때마다 이를 벡터화된 표현식으로 대체할 수 있는지 고려해야 합니다.

## NumPy의 Ufuncs 탐색

Ufuncs는 단일 입력에 대해 작동하는 *단항 ufuncs*와 두 개의 입력에 대해 작동하는 *이진 ufuncs*의 두 가지 형태로 존재합니다.
여기서는 이러한 두 가지 유형의 함수에 대한 예를 모두 살펴보겠습니다.

### 배열 연산

NumPy의 ufunc는 파이썬(Python)의 기본 산술 연산자를 사용하기 때문에 사용이 매우 자연스럽습니다.
표준 덧셈, 뺄셈, 곱셈, 나눗셈을 모두 사용할 수 있습니다.

In [7]:
x = np.arange(4)
print("x      =", x)
print("x + 5  =", x + 5)
print("x - 5  =", x - 5)
print("x * 2  =", x * 2)
print("x / 2  =", x / 2)
print("x // 2 =", x // 2)  # floor division

x      = [0 1 2 3]
x + 5  = [5 6 7 8]
x - 5  = [-5 -4 -3 -2]
x * 2  = [0 2 4 6]
x / 2  = [0.  0.5 1.  1.5]
x // 2 = [0 0 1 1]


부정을 위한 단항 ufunc, 지수를 위한 `**` 연산자, 모듈러스를 위한 `%` 연산자도 있습니다:

In [8]:
print("-x     = ", -x)
print("x ** 2 = ", x ** 2)
print("x % 2  = ", x % 2)

-x     =  [ 0 -1 -2 -3]
x ** 2 =  [0 1 4 9]
x % 2  =  [0 1 0 1]


또한 원하는 대로 연결할 수 있으며 표준 작업 순서가 준수됩니다.

In [9]:
-(0.5*x + 1) ** 2

array([-1.  , -2.25, -4.  , -6.25])

이러한 모든 산술 연산은 NumPy에 내장된 특정 ufunc를 둘러싼 편리한 래퍼일 뿐입니다. 예를 들어 `+` 연산자는 `add` ufunc에 대한 래퍼입니다.

In [10]:
np.add(x, 2)

array([2, 3, 4, 5])

다음 표에는 NumPy에서 구현된 산술 연산자가 나열되어 있습니다.

| 운영자 | 동등한 ufunc | 설명 |
|-------------|------|-----------|
|`+` |`np.add` |추가(예: `1 + 1 = 2`) |
|`-` |`np.subtract` |뺄셈(예: `3 - 2 = 1`) |
|`-` |`np.negative` |단항 부정(예: `-2`) |
|`*` |`np.multiply` |곱셈(예: `2 * 3 = 6`) |
|`/` |`np.divide` |나눗셈(예: `3 / 2 = 1.5`) |
|`//` |`np.floor_divide` |바닥 분할(예: `3 // 2 = 1`) |
|`**` |`np.power` |지수화(예: `2 ** 3 = 8`) |
|`%` |`np.mod` |계수/나머지 (예: `9 % 4 = 1`)|

또한 부울/비트 연산자도 있습니다. [비교, 마스크 및 부울 논리](02.06-Boolean-Arrays-and-Masks.ipynb)에서 이에 대해 살펴보겠습니다.

### 절대값

NumPy가 파이썬(Python)의 내장 산술 연산자를 이해하는 것처럼 파이썬(Python)의 내장 절대값 함수도 이해합니다.

In [11]:
x = np.array([-2, -1, 0, 1, 2])
abs(x)

array([2, 1, 0, 1, 2])

해당 NumPy ufunc는 `np.absolute`이며 `np.abs`라는 별칭으로도 사용할 수 있습니다.

In [12]:
np.absolute(x)

array([2, 1, 0, 1, 2])

In [13]:
np.abs(x)

array([2, 1, 0, 1, 2])

이 ufunc는 복잡한 데이터도 처리할 수 있으며, 이 경우 크기를 반환합니다.

In [14]:
x = np.array([3 - 4j, 4 - 3j, 2 + 0j, 0 + 1j])
np.abs(x)

array([5., 5., 2., 1.])

### 삼각 함수

NumPy는 수많은 유용한 ufunc를 제공하며, 데이터 과학(Data Science)자에게 가장 유용한 것 중 일부는 삼각 함수입니다.
각도 배열을 정의하는 것부터 시작하겠습니다.

In [15]:
theta = np.linspace(0, np.pi, 3)

이제 이러한 값에 대해 몇 가지 삼각 함수를 계산할 수 있습니다.

In [16]:
print("theta      = ", theta)
print("sin(theta) = ", np.sin(theta))
print("cos(theta) = ", np.cos(theta))
print("tan(theta) = ", np.tan(theta))

theta      =  [0.         1.57079633 3.14159265]
sin(theta) =  [0.0000000e+00 1.0000000e+00 1.2246468e-16]
cos(theta) =  [ 1.000000e+00  6.123234e-17 -1.000000e+00]
tan(theta) =  [ 0.00000000e+00  1.63312394e+16 -1.22464680e-16]


값은 기계 정밀도 내에서 계산되므로 0이어야 하는 값이 항상 정확히 0이 되는 것은 아닙니다.
역삼각 함수도 사용할 수 있습니다.

In [17]:
x = [-1, 0, 1]
print("x         = ", x)
print("arcsin(x) = ", np.arcsin(x))
print("arccos(x) = ", np.arccos(x))
print("arctan(x) = ", np.arctan(x))

x         =  [-1, 0, 1]
arcsin(x) =  [-1.57079633  0.          1.57079633]
arccos(x) =  [3.14159265 1.57079633 0.        ]
arctan(x) =  [-0.78539816  0.          0.78539816]


### 지수와 로그

NumPy ufuncs에서 사용할 수 있는 다른 일반적인 연산은 지수입니다.

In [18]:
x = [1, 2, 3]
print("x   =", x)
print("e^x =", np.exp(x))
print("2^x =", np.exp2(x))
print("3^x =", np.power(3., x))

x   = [1, 2, 3]
e^x = [ 2.71828183  7.3890561  20.08553692]
2^x = [2. 4. 8.]
3^x = [ 3.  9. 27.]


지수의 역함수인 로그도 사용할 수 있습니다.
기본 `np.log`는 자연 로그를 제공합니다. 밑이 2인 로그 또는 밑이 10인 로그를 계산하려는 경우 다음도 사용할 수 있습니다.

In [19]:
x = [1, 2, 4, 10]
print("x        =", x)
print("ln(x)    =", np.log(x))
print("log2(x)  =", np.log2(x))
print("log10(x) =", np.log10(x))

x        = [1, 2, 4, 10]
ln(x)    = [0.         0.69314718 1.38629436 2.30258509]
log2(x)  = [0.         1.         2.         3.32192809]
log10(x) = [0.         0.30103    0.60205999 1.        ]


매우 작은 입력으로 정밀도를 유지하는 데 유용한 몇 가지 특수 버전도 있습니다.

In [20]:
x = [0, 0.001, 0.01, 0.1]
print("exp(x) - 1 =", np.expm1(x))
print("log(1 + x) =", np.log1p(x))

exp(x) - 1 = [0.         0.0010005  0.01005017 0.10517092]
log(1 + x) = [0.         0.0009995  0.00995033 0.09531018]


`x`가 매우 작을 때 이 함수는 원시 `np.log` 또는 `np.exp`를 사용하는 경우보다 더 정확한 값을 제공합니다.

### 전문 Ufuncs

NumPy에는 쌍곡선 삼각법, 비트 산술, 비교 연산, 라디안에서 각도로의 변환, 반올림 및 나머지 등을 포함하여 더 많은 ufunc를 사용할 수 있습니다.
NumPy 문서를 살펴보면 흥미로운 기능이 많이 드러납니다.

보다 전문화된 ufunc에 대한 또 다른 훌륭한 소스는 하위 모듈 `scipy.special`입니다.
데이터에 대해 모호한 수학 함수를 계산하려는 경우 `scipy.special`에 구현될 가능성이 있습니다.
모두 나열하기에는 너무 많은 함수가 있지만 다음 스니펫은 통계 컨텍스트에서 나타날 수 있는 몇 가지를 보여줍니다.

In [21]:
from scipy import special

In [22]:
# Gamma functions (generalized factorials) and related functions
x = [1, 5, 10]
print("gamma(x)     =", special.gamma(x))
print("ln|gamma(x)| =", special.gammaln(x))
print("beta(x, 2)   =", special.beta(x, 2))

gamma(x)     = [1.0000e+00 2.4000e+01 3.6288e+05]
ln|gamma(x)| = [ 0.          3.17805383 12.80182748]
beta(x, 2)   = [0.5        0.03333333 0.00909091]


In [23]:
# Error function (integral of Gaussian),
# its complement, and its inverse
x = np.array([0, 0.3, 0.7, 1.0])
print("erf(x)  =", special.erf(x))
print("erfc(x) =", special.erfc(x))
print("erfinv(x) =", special.erfinv(x))

erf(x)  = [0.         0.32862676 0.67780119 0.84270079]
erfc(x) = [1.         0.67137324 0.32219881 0.15729921]
erfinv(x) = [0.         0.27246271 0.73286908        inf]


NumPy와 `scipy.special` 모두에서 사용할 수 있는 ufunc가 훨씬 더 많습니다.
이러한 패키지의 문서는 온라인에서 볼 수 있으므로 일반적으로 "gamma function python"을 웹 검색하면 관련 정보를 찾을 수 있습니다.

## 고급 Ufunc 기능

많은 NumPy 사용자는 전체 기능 세트를 배우지 않고도 ufunc를 사용합니다.
여기서는 ufuncs의 몇 가지 특수 기능을 간략하게 설명하겠습니다.

### 출력 지정

대규모 계산의 경우 계산 결과가 저장될 배열을 지정하는 것이 유용한 경우가 있습니다.
모든 ufunc에 대해 이는 함수의 'out' 인수를 사용하여 수행할 수 있습니다.

In [24]:
x = np.arange(5)
y = np.empty(5)
np.multiply(x, 10, out=y)
print(y)

[ 0. 10. 20. 30. 40.]


이는 배열 뷰에서도 사용할 수 있습니다. 예를 들어 지정된 배열의 다른 모든 요소에 계산 결과를 쓸 수 있습니다.

In [25]:
y = np.zeros(10)
np.power(2, x, out=y[::2])
print(y)

[ 1.  0.  2.  0.  4.  0.  8.  0. 16.  0.]


대신 `y[::2] = 2 ** x`를 작성했다면 `2 ** x`의 결과를 보관하기 위한 임시 배열이 생성된 후 해당 값을 `y` 배열에 복사하는 두 번째 작업이 수행되었을 것입니다.
이렇게 작은 계산에서는 큰 차이가 없지만 매우 큰 배열의 경우 'out' 인수를 주의 깊게 사용하면 메모리 절약 효과가 상당할 수 있습니다.

### 집계

바이너리 ufunc의 경우 개체에서 직접 집계를 계산할 수 있습니다.
예를 들어 특정 작업으로 배열을 *줄이기*하려면 ufunc의 `reduce` 메서드를 사용할 수 있습니다.
축소는 단일 결과만 남을 때까지 배열 요소에 지정된 작업을 반복적으로 적용합니다.

예를 들어 `add` ufunc에서 `reduce`를 호출하면 배열에 있는 모든 요소의 합계가 반환됩니다.

In [26]:
x = np.arange(1, 6)
np.add.reduce(x)

15

마찬가지로 `multiply` ufunc에서 `reduce`를 호출하면 모든 배열 요소의 곱이 생성됩니다.

In [27]:
np.multiply.reduce(x)

120

계산의 중간 결과를 모두 저장하려면 `accumulate`를 대신 사용할 수 있습니다.

In [28]:
np.add.accumulate(x)

array([ 1,  3,  6, 10, 15])

In [29]:
np.multiply.accumulate(x)

array([  1,   2,   6,  24, 120])

이러한 특별한 경우에는 결과를 계산하기 위한 전용 NumPy 함수(`np.sum`, `np.prod`, `np.cumsum`, `np.cumprod`)가 있으며, 이에 대해서는 [집계: 최소, 최대 및 그 사이의 모든 것](02.04-Computation-on-arrays-aggregates.ipynb)에서 살펴보겠습니다.

### 외부 제품

마지막으로, 모든 ufunc는 `외부` 방법을 사용하여 두 개의 서로 다른 입력의 모든 쌍의 출력을 계산할 수 있습니다.
이를 통해 한 줄로 구구단 생성과 같은 작업을 수행할 수 있습니다.

In [30]:
x = np.arange(1, 6)
np.multiply.outer(x, x)

array([[ 1,  2,  3,  4,  5],
       [ 2,  4,  6,  8, 10],
       [ 3,  6,  9, 12, 15],
       [ 4,  8, 12, 16, 20],
       [ 5, 10, 15, 20, 25]])

`ufunc.at` 및 `ufunc.reduceat` 방법도 유용하며 [Fancy Indexing](02.07-Fancy-Indexing.ipynb)에서 살펴보겠습니다.

또한 다양한 모양과 크기의 배열 사이에서 작동하는 ufunc의 기능, 즉 *브로드캐스팅*으로 알려진 일련의 작업을 접하게 됩니다.
이 주제는 전체 장을 다룰 만큼 중요합니다([배열에 대한 계산: 브로드캐스팅](02.05-Computation-on-arrays-broadcasting.ipynb) 참조).

## Ufuncs: 더 알아보기

범용 기능에 대한 자세한 내용(사용 가능한 기능의 전체 목록 포함)은 [NumPy](http://www.numpy.org) 및 [SciPy](http://www.scipy.org) 설명서 웹사이트에서 확인할 수 있습니다.

[I파이썬(Python)의 도움말과 문서](01.01-Help-And-Documentation.ipynb)에 설명된 대로 패키지를 가져오고 I파이썬(Python)의 탭 완성 및 도움말(`?`) 기능을 사용하여 I파이썬(Python) 내에서 직접 정보에 액세스할 수도 있습니다.